# Entrenamiento en Google Colab — CAMUS Segmentación

**Antes de correr:** Menú → Runtime → Change runtime type → **GPU (T4)**

Este notebook entrena los dos modelos y genera los resultados comparativos.

In [ ]:
# Celda 0 — Verificar GPU
import torch
print(f'GPU disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Celda 1 — Montar Drive y definir DRIVE_DIR
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/camus_checkpoints_v1'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Checkpoints se guardarán en: {DRIVE_DIR}')

In [ ]:
# Celda 2 — Clonar repo limpio
!rm -rf /content/camus-lv-segmentation
!git clone https://github.com/LourdesBranchi/camus-lv-segmentation.git /content/camus-lv-segmentation
%cd /content/camus-lv-segmentation
!pip install -r requirements.txt -q

In [ ]:
# Celda 3 — Agregar src al path
import sys
sys.path.insert(0, '/content/camus-lv-segmentation/src')

In [ ]:
# Celda 4 — Configurar Kaggle desde secret y descargar dataset
from google.colab import userdata
import os, json

kaggle_token = userdata.get('KAGGLE_API_TOKEN')
os.environ['KAGGLE_CONFIG_DIR'] = '/root/.kaggle'
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write(kaggle_token)
!chmod 600 /root/.kaggle/kaggle.json

!kaggle datasets download -d shoybhasan/camus-human-heart-data -q
!unzip -q camus-human-heart-data.zip -d /content/dataset_raw
!rm camus-human-heart-data.zip
print('Dataset descargado.')

In [ ]:
# Celda 5 — Descomprimir dataset interno y definir DATA_ROOT
import zipfile, os

inner = '/content/dataset_raw/download'
if os.path.exists(inner):
    with zipfile.ZipFile(inner, 'r') as z:
        z.extractall('/content/datos_corazon')
    print('Dataset listo.')
else:
    print('Buscando archivo comprimido interno:')
    !find /content/dataset_raw -name '*.zip' -o -name 'download'

DATA_ROOT = '/content/datos_corazon/database_nifti'
print(f'DATA_ROOT = {DATA_ROOT}')
!ls {DATA_ROOT} | head -5

In [ ]:
# Celda 6 — EDA rápido
from dataset import prepare_dataset
import matplotlib.pyplot as plt
import numpy as np

loaders = prepare_dataset(
    data_root=DATA_ROOT,
    batch_size=8,
    num_workers=2,
)

imgs, masks = next(iter(loaders['train']))
print(f'Batch imágenes: {imgs.shape}  dtype={imgs.dtype}')
print(f'Batch máscaras: {masks.shape}  clases={masks.unique().tolist()}')

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i in range(4):
    axes[0, i].imshow(imgs[i, 0].numpy(), cmap='gray')
    axes[0, i].set_title(f'Imagen {i+1}'); axes[0, i].axis('off')
    axes[1, i].imshow(masks[i].numpy(), cmap='jet', vmin=0, vmax=3)
    axes[1, i].set_title(f'Máscara {i+1}'); axes[1, i].axis('off')
plt.suptitle('Muestra del Dataset — Train Set', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Celda 7 — Entrenar Modelo 1: U-Net + ResNet34
from train import train, DEFAULT_CONFIG

config_unet = {**DEFAULT_CONFIG,
    'model':      'unet',
    'encoder':    'resnet34',
    'epochs':     50,
    'batch_size': 8,
    'lr':         1e-4,
    'patience':   10,
    'data_root':  DATA_ROOT,
    'save_dir':   DRIVE_DIR,
    'device':     'cuda',
}

history_unet = train(config_unet)

In [ ]:
# Celda 8 — Guardar history_unet como JSON en Drive
import json

unet_json_path = f'{DRIVE_DIR}/history_unet_resnet34.json'
with open(unet_json_path, 'w') as f:
    json.dump(history_unet, f)
print(f'history_unet guardado en: {unet_json_path}')

In [ ]:
# Celda 9 — Entrenar Modelo 2: Attention U-Net + EfficientNet-B0
config_attn = {**DEFAULT_CONFIG,
    'model':      'attention-unet',
    'encoder':    'efficientnet-b0',
    'epochs':     50,
    'batch_size': 8,
    'lr':         1e-4,
    'patience':   10,
    'data_root':  DATA_ROOT,
    'save_dir':   DRIVE_DIR,
    'device':     'cuda',
}

history_attn = train(config_attn)

In [ ]:
# Celda 10 — Guardar history_attn como JSON en Drive
import json

attn_json_path = f'{DRIVE_DIR}/history_attn_efficientnet.json'
with open(attn_json_path, 'w') as f:
    json.dump(history_attn, f)
print(f'history_attn guardado en: {attn_json_path}')

In [ ]:
# Celda 11 — Graficar curvas de training de ambos modelos
from evaluate import plot_training_history

plot_training_history(history_unet, 'U-Net + ResNet34',                  f'{DRIVE_DIR}/unet_history.png')
plot_training_history(history_attn, 'Attention U-Net + EfficientNet-B0', f'{DRIVE_DIR}/attn_history.png')

In [ ]:
# Celda 12 — Evaluar ambos modelos en test set y guardar resultados
import torch, json
from models   import get_model
from losses   import CombinedLoss
from evaluate import evaluate, print_metrics_table

device  = torch.device('cuda')
loss_fn = CombinedLoss()
results = {}

# U-Net
ckpt1 = torch.load(f'{DRIVE_DIR}/best_unet_resnet34.pth', map_location=device)
m1 = get_model('unet', encoder_name='resnet34').to(device)
m1.load_state_dict(ckpt1['model_state'])
r1 = evaluate(m1, loaders['test'], loss_fn, device, 'U-Net')
print_metrics_table(r1, 'U-Net + ResNet34')
results['U-Net + ResNet34'] = r1

# Attention U-Net
ckpt2 = torch.load(f'{DRIVE_DIR}/best_attention-unet_efficientnet-b0.pth', map_location=device)
m2 = get_model('attention-unet', encoder_name='efficientnet-b0').to(device)
m2.load_state_dict(ckpt2['model_state'])
r2 = evaluate(m2, loaders['test'], loss_fn, device, 'Attention U-Net')
print_metrics_table(r2, 'Attention U-Net + EfficientNet-B0')
results['Attention U-Net + EfficientNet-B0'] = r2

# Guardar resultados numéricos en Drive
with open(f'{DRIVE_DIR}/test_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'Resultados guardados en: {DRIVE_DIR}/test_results.json')

In [ ]:
# Celda 13 — visualize_predictions para ambos modelos
from evaluate import visualize_predictions

visualize_predictions(m1, loaders['test'], device,
                      n_samples=6, save_path=f'{DRIVE_DIR}/predictions_unet.png',
                      model_name='U-Net + ResNet34')

visualize_predictions(m2, loaders['test'], device,
                      n_samples=6, save_path=f'{DRIVE_DIR}/predictions_attn.png',
                      model_name='Attention U-Net + EfficientNet-B0')

In [ ]:
# Celda 14 — compare_models
from evaluate import compare_models

compare_models(results, save_path=f'{DRIVE_DIR}/comparativa_modelos.png')

print('\n' + '='*65)
print(f'{"Modelo":<40} {"LV":>6} {"MYO":>6} {"LA":>6} {"Mean":>6}')
print('='*65)
for name, m in results.items():
    print(f'{name:<40} {m["dice_lv"]:>6.3f} {m["dice_myo"]:>6.3f} {m["dice_la"]:>6.3f} {m["mean_dice_cardiac"]:>6.3f}')
print(f'{"Referencia (Leclerc 2019)":<40} {0.94:>6.3f} {0.85:>6.3f} {0.89:>6.3f} {0.89:>6.3f}')
print('='*65)

In [ ]:
# Celda 15 — Copiar todas las figuras y JSONs a Drive
import shutil, os

archivos_locales = [
    'predictions_unet.png',
    'predictions_attn.png',
    'comparativa_modelos.png',
    'unet_history.png',
    'attn_history.png',
]

for f in archivos_locales:
    if os.path.exists(f):
        shutil.copy(f, DRIVE_DIR)
        print(f'Copiado: {f} → {DRIVE_DIR}')
    else:
        print(f'No encontrado (ya guardado directo): {f}')

print(f'\n✓ Todos los resultados disponibles en: {DRIVE_DIR}')
!ls {DRIVE_DIR}